In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

df = pd.read_csv("cancer patient data sets.csv")  # Replace with your path

df.head()
df.info()
df.describe()

print("Missing Values:\n", df.isnull().sum())

# If missing values exist, we can handle them
# Here we drop rows with missing values (or could use imputation)
df.dropna(inplace=True)

duplicates = df.duplicated().sum()
print(f"Duplicate Records: {duplicates}")
df.drop_duplicates(inplace=True)  

categorical_cols = df.select_dtypes(include='object').columns
print("Categorical Columns:", categorical_cols)

for col in categorical_cols:
    dummies = pd.get_dummies(df[col], drop_first=True)  # Create dummy columns
    df = pd.concat([df.drop(col, axis=1), dummies], axis=1)  # Drop original and add dummies

sns.countplot(x="Label", data=df)
plt.title("Target Variable Distribution")
plt.show()

# Explanation: If dataset is imbalanced, we may need techniques like SMOTE or class_weight
# Here we just observe balance.

# Feature Correlation & Selection
corr_matrix = df.corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

# Pearson correlation with target
corr_target = corr_matrix["Label"].sort_values(ascending=False)
print("Correlation with Target:\n", corr_target)

# (Optional) Drop features with very low correlation with target
low_corr_features = corr_target[abs(corr_target) < 0.05].index
X = df.drop(columns=["Label"] + list(low_corr_features))
y = df["Label"]

X.hist(figsize=(12,8))
plt.show()

# If features have very different scales, apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Train-Test Split: 80%-20%
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0)

# Train-Validation Split: 70%-30% on training set
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=0)

# Explanation: Validation set is used to tune hyperparameters and avoid overfitting

# ----- Default Gini -----
dt_default = DecisionTreeClassifier(random_state=0)
dt_default.fit(X_train, y_train)
pred_default = dt_default.predict(X_val)
print("\n========= Decision Tree (Default Gini) =========")
print("Training Accuracy:", dt_default.score(X_train, y_train) * 100, "%")
print("Validation Accuracy:", accuracy_score(y_val, pred_default) * 100, "%")

# ----- Entropy -----
dt_entropy = DecisionTreeClassifier(criterion='entropy', random_state=0)
dt_entropy.fit(X_train, y_train)
pred_entropy = dt_entropy.predict(X_val)
print("\n========= Decision Tree (Entropy) =========")
print("Training Accuracy:", dt_entropy.score(X_train, y_train) * 100, "%")
print("Validation Accuracy:", accuracy_score(y_val, pred_entropy) * 100, "%")

# ----- Entropy + Pruning -----
dt_pruned = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.015, random_state=0)
dt_pruned.fit(X_train, y_train)
pred_pruned = dt_pruned.predict(X_val)
print("\n========= Decision Tree (Entropy + Pruning) =========")
print("Training Accuracy:", dt_pruned.score(X_train, y_train) * 100, "%")
print("Validation Accuracy:", accuracy_score(y_val, pred_pruned) * 100, "%")

# ==========================
# Step 8: Decision Tree Visualization
# ==========================
plt.figure(figsize=(12,8))
plot_tree(dt_pruned, filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Visualization (Entropy + Pruning)")
plt.show()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   index                     1000 non-null   int64 
 1   Patient Id                1000 non-null   object
 2   Age                       1000 non-null   int64 
 3   Gender                    1000 non-null   int64 
 4   Air Pollution             1000 non-null   int64 
 5   Alcohol use               1000 non-null   int64 
 6   Dust Allergy              1000 non-null   int64 
 7   OccuPational Hazards      1000 non-null   int64 
 8   Genetic Risk              1000 non-null   int64 
 9   chronic Lung Disease      1000 non-null   int64 
 10  Balanced Diet             1000 non-null   int64 
 11  Obesity                   1000 non-null   int64 
 12  Smoking                   1000 non-null   int64 
 13  Passive Smoker            1000 non-null   int64 
 14  Chest Pain               

ValueError: Could not interpret value `Label` for `x`. An entry with this name does not appear in `data`.